In [ ]:
%pip install -q openai-whisper google-genai pydantic pandas rapidfuzz jiwer openpyxl python-docx ipywidgets

print("✅ Installation completed.")

In [ ]:
import sys
import shutil
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import torch
import whisper
import pandas as pd
import pydantic
import ipywidgets as widgets

from google import genai
from google.genai import types
from jiwer import process_words, cer
from IPython.display import display, Audio

print("Python:", sys.version.split()[0])
print("Pydantic:", pydantic.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "CPU"
)
print("FFmpeg:", shutil.which("ffmpeg"))
print("✅ Packages imported successfully.")


In [ ]:
PROJECT_ROOT = Path("/content/AIMIS_Text_Audio_Evaluation")

DIRS = {
    "inputs": PROJECT_ROOT / "inputs",
    "raw_transcripts": PROJECT_ROOT / "transcripts" / "raw",
    "reviewed_transcripts": PROJECT_ROOT / "transcripts" / "reviewed",
    "annotations": PROJECT_ROOT / "annotations",
    "generic_outputs": PROJECT_ROOT / "outputs" / "generic",
    "structured_outputs": PROJECT_ROOT / "outputs" / "structured",
    "evaluation_evidence": PROJECT_ROOT / "evaluation_evidence",
    "results": PROJECT_ROOT / "results",
    "logs": PROJECT_ROOT / "logs",
}

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Text and audio project ready.")
print("Location:", PROJECT_ROOT)

In [ ]:
manifest_path = PROJECT_ROOT / "meeting_register.csv"

columns = [
    "meeting_id",
    "topic",
    "source_type",
    "input_type",
    "input_filename",
    "raw_transcript_filename",
    "reviewed_transcript_filename",
    "transcript_review_status",
    "annotation_status",
    "generic_status",
    "structured_status",
    "evaluation_status",
]

if manifest_path.exists():
    meeting_register = pd.read_csv(
        manifest_path, dtype=str, keep_default_na=False
    )
else:
    meeting_register = pd.DataFrame(columns=columns)
    meeting_register.to_csv(manifest_path, index=False)

print("✅ Combined meeting register ready.")
print("Registered meetings:", len(meeting_register))

In [ ]:
from google.colab import files
import hashlib

uploaded = files.upload()

audio_extensions = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".mp4"}
text_extensions = {".txt", ".docx"}

existing_hashes = {
    hashlib.sha256(path.read_bytes()).hexdigest()
    for path in DIRS["inputs"].iterdir()
    if path.is_file()
}

pending = []
seen_hashes = set(existing_hashes)

for filename, content in sorted(uploaded.items()):
    extension = Path(filename).suffix.lower()

    if extension not in audio_extensions | text_extensions:
        raise ValueError(f"Unsupported file: {filename}")

    digest = hashlib.sha256(content).hexdigest()
    if digest in seen_hashes:
        raise ValueError(f"Duplicate file detected: {filename}")

    seen_hashes.add(digest)
    pending.append((filename, content, extension))

next_number = (
    max(
        [int(mid[1:]) for mid in meeting_register["meeting_id"]],
        default=0
    ) + 1
)

new_rows = []

for number, (filename, content, extension) in enumerate(
    pending, start=next_number
):
    meeting_id = f"M{number:02d}"
    saved_name = f"{meeting_id}{extension}"
    path = DIRS["inputs"] / saved_name

    if path.exists():
        raise FileExistsError(f"Input already exists: {saved_name}")

    path.write_bytes(content)

    row = {column: "" for column in columns}
    row.update({
        "meeting_id": meeting_id,
        "input_type": (
            "audio" if extension in audio_extensions else "text"
        ),
        "input_filename": saved_name,
        "transcript_review_status": "pending",
        "annotation_status": "pending",
        "generic_status": "pending",
        "structured_status": "pending",
        "evaluation_status": "pending",
    })
    new_rows.append(row)
    print(f"✅ {meeting_id} ← {filename} ({row['input_type']})")

if new_rows:
    meeting_register = pd.concat(
        [meeting_register, pd.DataFrame(new_rows)],
        ignore_index=True
    )
    meeting_register.to_csv(manifest_path, index=False)

print("Total registered meetings:", len(meeting_register))

In [ ]:
from docx import Document

asr_model = None

for index, row in meeting_register.iterrows():
    mid = row["meeting_id"]
    input_path = DIRS["inputs"] / row["input_filename"]
    raw_path = DIRS["raw_transcripts"] / f"{mid}_raw.txt"
    log_path = DIRS["logs"] / f"{mid}_input_processing.json"

    if raw_path.exists():
        meeting_register.loc[
            index, "raw_transcript_filename"
        ] = raw_path.name
        print(f"✅ {mid}: saved text reused.")
        continue

    started = time.perf_counter()
    log = {
        "meeting_id": mid,
        "input_type": row["input_type"],
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    try:
        if row["input_type"] == "audio":
            if asr_model is None:
                print("Loading Whisper small...")
                device = "cuda" if torch.cuda.is_available() else "cpu"
                asr_model = whisper.load_model("small", device=device)

            result = asr_model.transcribe(
                str(input_path),
                language="en",
                temperature=0,
                fp16=torch.cuda.is_available(),
                verbose=False
            )
            text = result["text"].strip()

            (DIRS["raw_transcripts"] / f"{mid}_whisper.json"
            ).write_text(
                json.dumps(result, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )
            log["whisper_model"] = "small"
            log["language"] = "en"

        elif input_path.suffix.lower() == ".docx":
            document = Document(input_path)
            text = "\n".join(
                paragraph.text for paragraph in document.paragraphs
                if paragraph.text.strip()
            )
        else:
            text = input_path.read_text(encoding="utf-8-sig")

        if not text.strip():
            raise ValueError("No text extracted.")

        raw_path.write_text(text, encoding="utf-8")
        meeting_register.loc[
            index, "raw_transcript_filename"
        ] = raw_path.name

        log["status"] = "success"
        log["words"] = len(text.split())
        print(f"✅ {mid} ({row['input_type']}): {log['words']} words.")

    except Exception as error:
        log["status"] = "failed"
        log["error"] = str(error)
        print(f"❌ {mid}: {error}")

    finally:
        log["processing_seconds"] = round(
            time.perf_counter() - started, 3
        )
        log_path.write_text(
            json.dumps(log, indent=2), encoding="utf-8"
        )
        meeting_register.to_csv(manifest_path, index=False)

print("✅ Input-processing batch finished.")

In [ ]:
import re
from itertools import combinations
from rapidfuzz.fuzz import ratio

def comparison_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

texts = {}

for _, row in meeting_register.iterrows():
    path = DIRS["raw_transcripts"] / row["raw_transcript_filename"]
    texts[row["meeting_id"]] = comparison_text(
        path.read_text(encoding="utf-8")
    )

possible_duplicates = []

for first, second in combinations(texts, 2):
    similarity = ratio(texts[first], texts[second])

    if similarity >= 65:
        possible_duplicates.append({
            "first": first,
            "second": second,
            "text_similarity_percent": round(similarity, 2)
        })

duplicate_candidates = pd.DataFrame(
    possible_duplicates,
    columns=["first", "second", "text_similarity_percent"]
)

duplicate_candidates.to_csv(
    DIRS["evaluation_evidence"] / "duplicate_candidates.csv",
    index=False
)

if duplicate_candidates.empty:
    print("No highly similar pairs flagged.")
else:
    display(duplicate_candidates)

print("Similarity is a review aid, not a final duplicate decision.")

In [ ]:
proposed_groups = {
    "M01": "C01",
    "M03": "C01",
    "M04": "C01",
    "M02": "C02",
    "M05": "C02",
}

for number in range(6, 17):
    proposed_groups[f"M{number:02d}"] = f"C{number - 3:02d}"

print("Proposed repeated-conversation groups:")
print("C01: M01 audio, M03 text, M04 audio")
print("C02: M02 text, M05 audio")

confirmed = input(
    "Do these groups contain the same conversations? yes/no: "
).strip().lower()

if confirmed == "yes":
    meeting_register["conversation_id"] = (
        meeting_register["meeting_id"].map(proposed_groups)
    )
    meeting_register.to_csv(manifest_path, index=False)

    print("✅ Conversation links saved.")
    print(
        "Distinct conversations:",
        meeting_register["conversation_id"].nunique()
    )
else:
    print("Links not saved. Tell me which groups differ.")

In [ ]:
from google.colab import files
import shutil
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

backup_path = shutil.make_archive(
    f"/content/AIMIS_Text_Audio_Backup_{stamp}",
    "zip",
    root_dir=str(PROJECT_ROOT.parent),
    base_dir=PROJECT_ROOT.name
)

files.download(backup_path)
print("✅ Backup ready. Keep the downloaded ZIP.")

In [ ]:
selector = widgets.Dropdown(
    options=meeting_register["meeting_id"].tolist(),
    description="Input:"
)
topic = widgets.Text(description="Topic:")
source = widgets.Dropdown(
    options=["Select source", "real", "scripted", "synthetic"],
    description="Source:"
)
reviewer = widgets.Text(value="wajahat", description="Reviewer:")
editor = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="320px")
)
checked = widgets.Checkbox(
    description="I checked the complete input.", value=False
)
save = widgets.Button(description="Save reviewed transcript")
player = widgets.Output()
message = widgets.Output()

def load_review(change=None):
    mid = selector.value
    row = meeting_register[
        meeting_register["meeting_id"] == mid
    ].iloc[0]

    reviewed = DIRS["reviewed_transcripts"] / f"{mid}_reviewed.txt"
    raw = DIRS["raw_transcripts"] / row["raw_transcript_filename"]

    editor.value = (
        reviewed if reviewed.exists() else raw
    ).read_text(encoding="utf-8")

    topic.value = row["topic"]
    source.value = (
        row["source_type"]
        if row["source_type"] in ["real", "scripted", "synthetic"]
        else "Select source"
    )
    checked.value = False
    message.clear_output()

    with player:
        player.clear_output()
        print("Conversation:", row["conversation_id"])
        print("Input type:", row["input_type"])
        if row["input_type"] == "audio":
            display(Audio(filename=str(
                DIRS["inputs"] / row["input_filename"]
            )))
        else:
            print("Compare the text with your original document.")

def save_review(_):
    with message:
        message.clear_output()

        if (
            not checked.value
            or not reviewer.value.strip()
            or not topic.value.strip()
            or not editor.value.strip()
            or source.value == "Select source"
        ):
            print("Complete the fields and tick the review box.")
            return

        mid = selector.value
        path = DIRS["reviewed_transcripts"] / f"{mid}_reviewed.txt"
        path.write_text(editor.value.strip(), encoding="utf-8")

        meeting_register.loc[
            meeting_register["meeting_id"] == mid,
            ["topic", "source_type", "reviewed_transcript_filename",
             "transcript_review_status"]
        ] = [
            topic.value.strip(), source.value,
            path.name, "human_reviewed"
        ]

        log = {
            "meeting_id": mid,
            "reviewer": reviewer.value.strip(),
            "reviewed_at": datetime.now(timezone.utc).isoformat(),
            "status": "human_reviewed",
            "method": "Checked against original input"
        }
        (DIRS["logs"] / f"{mid}_transcript_review.json").write_text(
            json.dumps(log, indent=2), encoding="utf-8"
        )
        meeting_register.to_csv(manifest_path, index=False)
        print(f"✅ {mid} reviewed transcript saved.")

selector.observe(load_review, names="value")
save.on_click(save_review)

display(
    selector, player, topic, source, reviewer,
    editor, checked, save, message
)
load_review()

In [ ]:
meeting_register = pd.read_csv(
    manifest_path, dtype=str, keep_default_na=False
)

pending = meeting_register[
    meeting_register["transcript_review_status"] != "human_reviewed"
]

print(
    "Reviewed inputs:",
    len(meeting_register) - len(pending),
    "/", len(meeting_register)
)

if pending.empty:
    print("✅ All transcript reviews recorded.")
else:
    print("Still pending:")
    display(pending[["meeting_id", "input_type"]])

In [ ]:
import re
from jiwer import process_words, cer

def normalize_asr(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

asr_rows = []

for _, row in meeting_register.iterrows():
    result = {
        "meeting_id": row["meeting_id"],
        "conversation_id": row["conversation_id"],
        "input_type": row["input_type"],
        "reference_words": None,
        "substitutions": None,
        "deletions": None,
        "insertions": None,
        "WER_percent": None,
        "CER_percent": None,
        "reference_method": "not applicable",
    }

    if row["input_type"] == "audio":
        raw = normalize_asr(
            (DIRS["raw_transcripts"] / row["raw_transcript_filename"])
            .read_text(encoding="utf-8")
        )
        reference = normalize_asr(
            (DIRS["reviewed_transcripts"]
             / row["reviewed_transcript_filename"])
            .read_text(encoding="utf-8")
        )

        if not reference:
            raise ValueError(f"{row['meeting_id']}: empty reference.")

        comparison = process_words(reference, raw)

        result.update({
            "reference_words": len(reference.split()),
            "substitutions": comparison.substitutions,
            "deletions": comparison.deletions,
            "insertions": comparison.insertions,
            "WER_percent": round(100 * comparison.wer, 2),
            "CER_percent": round(100 * cer(reference, raw), 2),
            "reference_method": "human-reviewed Whisper transcript",
        })

    asr_rows.append(result)

asr_results = pd.DataFrame(asr_rows)
asr_results.to_csv(
    DIRS["results"] / "transcription_metrics.csv",
    index=False
)

display(asr_results)
print("WER = (substitutions + deletions + insertions)")
print("      ÷ reference words × 100")

In [ ]:
conversation_inputs = {}

for conversation_id, group in meeting_register.groupby(
    "conversation_id", sort=True
):
    text_inputs = group[group["input_type"] == "text"]
    selected = (
        text_inputs.iloc[0]
        if not text_inputs.empty
        else group.iloc[0]
    )

    conversation_inputs[conversation_id] = selected["meeting_id"]

    transcript = (
        DIRS["reviewed_transcripts"]
        / selected["reviewed_transcript_filename"]
    ).read_text(encoding="utf-8")

    print(
        f"\n===== {conversation_id} "
        f"(reference input: {selected['meeting_id']}) ====="
    )
    print(transcript)

(
    PROJECT_ROOT / "conversation_reference_inputs.json"
).write_text(
    json.dumps(conversation_inputs, indent=2),
    encoding="utf-8"
)

In [ ]:
annotation_rules = {
    "version": "1.0",
    "decisions": (
        "Include explicitly accepted choices or manager instructions. "
        "Exclude suggestions without clear acceptance."
    ),
    "actions": (
        "Include future assignments and accepted commitments. "
        "Exclude completed work and general aspirations."
    ),
    "task_units": (
        "Separate independently executable tasks. Keep a report "
        "and its required contents together. Separate preparing "
        "and sharing a deliverable."
    ),
    "owners": (
        "Use explicitly named owners. For 'we', record the speaker "
        "and mark collective responsibility as ambiguous unless "
        "the responsible group is explicitly identified."
    ),
    "deadlines": (
        "Preserve spoken wording. Keep missing deadlines blank. "
        "Do not apply one task's deadline to another task."
    ),
    "conditions": (
        "Record conditional triggers separately from deadlines."
    ),
    "evidence": (
        "Include transcript quotes and surrounding acceptance "
        "where needed. Do not invent supporting quotations."
    ),
    "duplicates": (
        "Record each distinct task once; retain repeated mentions "
        "as supporting evidence."
    ),
    "review": (
        "AI-assisted drafts require human review before evaluation."
    )
}

(PROJECT_ROOT / "annotation_rules.json").write_text(
    json.dumps(annotation_rules, indent=2),
    encoding="utf-8"
)

print("✅ Annotation rules saved.")
print("13 distinct conversations will receive reference annotations.")

In [ ]:
# Decisions: task, supporting quote
# Actions: task, owner, deadline, condition, supporting quote

reference_data = {
    "C01": {
        "decisions": [
            ("Flag incomplete responses and defer the missing-data treatment decision.",
             "For now, just flag them."),
            ("Use employee participation consistently.",
             "Agreed. Let’s keep “employee participation” throughout the report."),
            ("Proceed with available data and document the limitation if the department does not respond by tomorrow afternoon.",
             "If they still haven’t responded by the afternoon, we’ll proceed with what we have and document that limitation.")
        ],
        "actions": [
            ("Send another reminder.", "James", "",
             "If there is still no response by tomorrow morning.",
             "Maybe give them until tomorrow morning, and if there’s still no response, send another reminder."),
            ("Continue cleaning existing data.", "Aisha", "", "",
             "Aisha will continue cleaning and analyzing the existing data"),
            ("Continue analyzing existing data.", "Aisha", "", "",
             "Aisha will continue cleaning and analyzing the existing data"),
            ("Flag incomplete responses.", "Aisha", "", "",
             "she’ll flag the incomplete responses for now."),
            ("Continue working on the report.", "Michael", "", "",
             "Michael will continue working on the report"),
            ("Replace inconsistent terminology with employee participation.",
             "Michael", "", "",
             "Michael, can you change the other references to match?"),
            ("Send Sarah an updated draft.", "Michael", "tomorrow afternoon", "",
             "I should be able to send you an updated draft tomorrow afternoon."),
            ("Review the updated report.", "Sarah", "tomorrow evening",
             "Once Michael sends the updated version.",
             "I’ll also review the report tomorrow evening once Michael sends the updated version.")
        ]
    },
    "C02": {
        "decisions": [
            ("Complete documentation before final release.",
             "It is important that the documentation is completed before the final release."),
            ("Do not rush final testing; prioritize quality.",
             "I agree. Quality should always be our priority."),
            ("Arrange a selected-user feedback session and document feedback.",
             "That is a good idea. Please arrange that session and make sure the feedback is properly documented.")
        ],
        "actions": [
            ("Complete remaining technical improvements.", "Ali", "", "",
             "Ali will complete the remaining technical improvements"),
            ("Resolve identified technical issues.", "Ali", "", "",
             "resolve the identified issues."),
            ("Provide a progress update.", "Ali", "",
             "After completing technical improvements.",
             "I will provide a progress update after completing the technical improvements."),
            ("Complete the quality review.", "Hassan", "", "",
             "I will complete the quality review"),
            ("Update documentation requirements.", "Hassan", "", "",
             "update the documentation requirements."),
            ("Share quality-review observations.", "Hassan", "", "",
             "share my observations with the team."),
            ("Coordinate additional testing support.", "Sarah", "", "",
             "Sarah, can you coordinate with the testing department and arrange additional support?"),
            ("Prepare a testing schedule.", "Sarah", "", "",
             "I will contact the testing department and prepare a schedule."),
            ("Update the project timeline.", "Sarah", "", "",
             "update the project timeline."),
            ("Share the updated project timeline.", "Sarah", "", "",
             "I will also share an updated project timeline with all team members."),
            ("Arrange a selected-user feedback session.", "Sarah", "next week", "",
             "We can arrange a feedback session with selected users next week."),
            ("Ensure feedback is documented.", "Sarah", "", "",
             "make sure the feedback is properly documented."),
            ("Prepare the feedback-session user list.", "Sarah", "", "",
             "I will prepare the list of users"),
            ("Share the feedback-session schedule.", "Sarah", "", "",
             "share the meeting schedule."),
            ("Coordinate with all departments.", "Sarah", "", "",
             "I will coordinate with all departments")
        ]
    },
    "C03": {
        "decisions": [
            ("Include leadership development in the training plan.",
             "That is a good suggestion. Leadership development can help employees take more responsibility and prepare them for future roles."),
            ("Prepare a detailed six-month training plan.",
             "Excellent Please prepare a detailed training plan with objectives, schedule, estimated cost and expected outcomes.")
        ],
        "actions": [
            ("Prepare the detailed training plan with objectives, schedule, cost and expected outcomes.",
             "Aisha", "", "",
             "Please prepare a detailed training plan with objectives, schedule, estimated cost and expected outcomes."),
            ("Share the first training-plan draft.", "Aisha", "within one week", "",
             "I will prepare the plan and share the first draft within one week."),
            ("Coordinate with department managers to identify training participants.",
             "Mr. Bilal", "", "",
             "I will coordinate with department managers to identify employees who should participate in each training session."),
            ("Collect employee feedback.", "Usman", "", "",
             "I will collect feedback from employees"),
            ("Share employee suggestions with the training team.", "Usman", "", "",
             "share their suggestions with the training team.")
        ]
    },
    "C04": {
        "decisions": [
            ("Support limited-time promotions subject to long-term strategy and product-value safeguards.",
             "that is a good idea. However, we need to ensure that promotions support our long-term strategy and do not reduce the perceived value of our products."),
            ("Include customer feedback in the regular marketing process.",
             "Customer feedback should become part of our regular marketing process.")
        ],
        "actions": [
            ("Prepare a sales-trends report identifying conversion improvements.",
             "Sara", "", "",
             "I will prepare a report on current sales trends and identify areas where we can improve customer conversion."),
            ("Prepare a digital marketing plan covering social media and website enhancements.",
             "Ahmed", "", "",
             "I will prepare a digital marketing improvement plan, including social media strategies and website enhancements."),
            ("Organize the latest customer feedback.", "Nadia", "", "",
             "I will organize the latest customer feedback"),
            ("Prepare a summary of customer concerns and suggestions.", "Nadia", "", "",
             "prepare a summary of common concerns and suggestions."),
            ("Share the digital strategy draft.", "Ahmed", "soon", "",
             "I will share the digital strategy draft soon."),
            ("Provide the customer feedback report.", "Nadia", "", "",
             "I will provide the customer feedback report.")
        ]
    },
    "C05": {
        "decisions": [
            ("Use a progress tracking system.",
             "that is a good suggestion. Please prepare the tracking system and share it with the team.")
        ],
        "actions": [
            ("Prepare the tracking system.", "Hina", "", "",
             "Please prepare the tracking system"),
            ("Share the tracking system.", "Hina", "", "",
             "share it with the team."),
            ("Coordinate with all departments.", "Sara", "", "",
             "I will coordinate with all departments"),
            ("Distribute the updated schedule.", "Sara", "", "",
             "make sure everyone receives the updated schedule."),
            ("Start assigned tasks.", "Bill", "", "",
             "we will start the assigned tasks"),
            ("Provide weekly progress updates.", "Bill", "weekly", "",
             "provide weekly progress updates.")
        ]
    },
    "C06": {
        "decisions": [
            ("Adopt monthly departmental expense reporting.",
             "That will help us make better financial decisions. Please prepare the reporting format.")
        ],
        "actions": [
            ("Prepare the expense-reporting format.", "Meriam", "", "",
             "Please prepare the reporting format."),
            ("Prepare the financial summary.", "Humor", "", "",
             "I will prepare the financial summary"),
            ("Share the financial summary.", "Humor", "", "",
             "share it with everyone."),
            ("Review departmental requirements.", "Zain", "", "",
             "We will review our requirements"),
            ("Prioritize important expenses.", "Zain", "", "",
             "prioritize important expenses.")
        ]
    },
    "C07": {
        "decisions": [
            ("Improve response time and staff training.",
             "we need to improve response time and provide better training to staff.")
        ],
        "actions": [
            ("Introduce a customer-query response tracking system.", "Fatima", "", "",
             "we will introduce a response tracking system to monitor customer queries."),
            ("Prepare a quality improvement report from recent feedback.", "Ahmed", "", "",
             "I will prepare a quality improvement report based on recent feedback.")
        ]
    },
    "C08": {
        "decisions": [
            ("Use technical and communication skills as selection criteria.",
             "we should focus on candidates with both technical skills and good communication abilities. HR manager, agreed."),
            ("Use fair selection based on clear criteria.",
             "The selection process should be fair and based on clear criteria.")
        ],
        "actions": [
            ("Prepare a candidate shortlist.", "Recruitment officer", "", "",
             "I will prepare a shortlist"),
            ("Arrange interviews.", "Recruitment officer", "next week", "",
             "arrange interviews next week."),
            ("Provide technical interview questions.", "Department head", "", "",
             "we will provide technical questions for the interview process.")
        ]
    },
    "C09": {
        "decisions": [
            ("Include practical exercises and additional learning materials in future training.",
             "we can include practical exercises and additional learning materials in future sessions. Training manager, that is a valuable suggestion.")
        ],
        "actions": [
            ("Update training content with practical demonstrations.", "Instructor", "", "",
             "I will update the training content and include more practical demonstrations."),
            ("Collect participant feedback.", "Coordinator", "", "",
             "I will collect participant feedback"),
            ("Prepare the final evaluation report.", "Coordinator", "", "",
             "prepare a final evaluation report.")
        ]
    },
    "C10": {
        "decisions": [
            ("Provide employee training before implementation.",
             "We also need to ensure employees receive proper training before implementation.")
        ],
        "actions": [
            ("Arrange employee training sessions.", "Mr. Ahmed", "before implementation", "",
             "We will arrange training sessions"),
            ("Provide user guides.", "Mr. Ahmed", "", "",
             "provide user guides."),
            ("Prepare the implementation schedule.", "", "", "",
             "Please prepare the implementation schedule"),
            ("Identify implementation risks.", "", "", "",
             "identify possible risks."),
            ("Coordinate with all departments.", "Sara", "", "",
             "We will coordinate with all departments"),
            ("Share the final plan.", "Sara", "", "",
             "share the final plan.")
        ]
    },
    "C11": {
        "decisions": [
            ("Prepare an academic improvement plan.",
             "these are useful suggestions. Please prepare an academic improvement plan.")
        ],
        "actions": [
            ("Prepare the academic improvement plan.", "Ayesha", "", "",
             "we will prepare the plan"),
            ("Submit the academic improvement plan.", "Ayesha", "next week", "",
             "submit it next week.")
        ]
    },
    "C12": {
        "decisions": [
            ("Increase safety awareness among employees.",
             "Agreed. Safety awareness should be increased among all employees.")
        ],
        "actions": [
            ("Prepare a safety training schedule.", "Mr. Ahmed", "", "",
             "I will prepare a training schedule"),
            ("Prepare a safety checklist.", "Mr. Ahmed", "", "",
             "safety checklist.")
        ]
    },
    "C13": {
        "decisions": [
            ("Improve financial monitoring through regular reporting.",
             "We can improve financial monitoring through regular reporting. Mr. Yuma, that is a good suggestion.")
        ],
        "actions": [
            ("Prepare a detailed budget review report.", "Fatima", "", "",
             "I will prepare a detailed budget review report.")
        ]
    }
}

# Flag collective commitments without inventing team names.
collective_owners = {
    "C05": ["A5", "A6"],
    "C06": ["A4", "A5"],
    "C07": ["A1"],
    "C08": ["A3"],
    "C10": ["A1", "A2", "A5", "A6"],
    "C11": ["A1", "A2"]
}

conversation_drafts = {}

for cid, data in reference_data.items():
    reviewed_path = DIRS["annotations"] / f"{cid}_reference_reviewed.json"
    draft_path = DIRS["annotations"] / f"{cid}_reference_draft.json"

    if reviewed_path.exists() or draft_path.exists():
        print(f"{cid}: existing annotations preserved.")
        continue

    draft = {
        "conversation_id": cid,
        "reference_input": conversation_inputs[cid],
        "rules_version": "1.0",
        "annotation_origin": "AI-assisted draft",
        "status": "pending_human_review",
        "decisions": [
            {
                "id": f"D{i}",
                "task": task,
                "evidence": quote
            }
            for i, (task, quote)
            in enumerate(data["decisions"], 1)
        ],
        "action_items": [
            {
                "id": f"A{i}",
                "task": task,
                "owner": owner,
                "owner_ambiguous": (
                    f"A{i}" in collective_owners.get(cid, [])
                    or not owner
                ),
                "deadline": deadline,
                "condition": condition,
                "evidence": quote
            }
            for i, (task, owner, deadline, condition, quote)
            in enumerate(data["actions"], 1)
        ]
    }

    draft_path.write_text(
        json.dumps(draft, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    conversation_drafts[cid] = draft

    print(
        f"✅ {cid}: {len(draft['decisions'])} decisions, "
        f"{len(draft['action_items'])} actions."
    )

print("Drafts saved. Human review required.")

In [ ]:
annotation_selector = widgets.Dropdown(
    options=sorted(conversation_inputs),
    description="Conversation:",
    style={"description_width": "initial"}
)
annotation_transcript = widgets.Output()
annotation_editor = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="450px")
)
annotation_reviewer = widgets.Text(
    value="wajahat", description="Reviewer:"
)
annotation_checked = widgets.Checkbox(
    value=False,
    description="I checked every annotation against the transcript."
)
annotation_save = widgets.Button(description="Save reviewed annotations")
annotation_message = widgets.Output()

def load_annotation(change=None):
    cid = annotation_selector.value
    reviewed = DIRS["annotations"] / f"{cid}_reference_reviewed.json"
    draft = DIRS["annotations"] / f"{cid}_reference_draft.json"

    annotation_editor.value = (
        reviewed if reviewed.exists() else draft
    ).read_text(encoding="utf-8")

    annotation_checked.value = False
    annotation_message.clear_output()

    mid = conversation_inputs[cid]
    row = meeting_register[
        meeting_register["meeting_id"] == mid
    ].iloc[0]

    with annotation_transcript:
        annotation_transcript.clear_output()
        print(
            (DIRS["reviewed_transcripts"]
             / row["reviewed_transcript_filename"])
            .read_text(encoding="utf-8")
        )

def save_annotation(_):
    with annotation_message:
        annotation_message.clear_output()

        if (
            not annotation_checked.value
            or not annotation_reviewer.value.strip()
        ):
            print("Enter reviewer and tick the review box.")
            return

        try:
            data = json.loads(annotation_editor.value)
            cid = annotation_selector.value

            if data["conversation_id"] != cid:
                raise ValueError("Conversation ID must match the selection.")

            for key, prefix in [
                ("decisions", "D"), ("action_items", "A")
            ]:
                items = data[key]
                if not isinstance(items, list):
                    raise ValueError(f"{key} must be a list.")

                for i, item in enumerate(items, 1):
                    item["id"] = f"{prefix}{i}"
                    if not item["task"].strip() or not item["evidence"].strip():
                        raise ValueError("Every item needs a task and evidence.")

                    if key == "action_items":
                        for field in ["owner", "deadline", "condition"]:
                            if not isinstance(item[field], str):
                                raise ValueError(f"{field} must be text.")
                        if not isinstance(item["owner_ambiguous"], bool):
                            raise ValueError("owner_ambiguous must be true or false.")

            data.update({
                "status": "human_reviewed",
                "reviewer": annotation_reviewer.value.strip(),
                "reviewed_at": datetime.now(timezone.utc).isoformat()
            })

            path = DIRS["annotations"] / f"{cid}_reference_reviewed.json"
            path.write_text(
                json.dumps(data, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )

            print(f"✅ {cid} reference annotations reviewed and saved.")

        except Exception as error:
            print("Not saved:", error)

annotation_selector.observe(load_annotation, names="value")
annotation_save.on_click(save_annotation)

display(
    annotation_selector, annotation_transcript,
    annotation_editor, annotation_reviewer,
    annotation_checked, annotation_save, annotation_message
)
load_annotation()

In [ ]:
pending = []

for cid in sorted(conversation_inputs):
    path = DIRS["annotations"] / f"{cid}_reference_reviewed.json"

    if not path.exists():
        pending.append(cid)
        continue

    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("status") != "human_reviewed":
        pending.append(cid)

print("Reviewed conversations:", 13 - len(pending), "/ 13")

if pending:
    print("Still pending:", ", ".join(pending))
else:
    meeting_register["annotation_status"] = "human_reviewed"
    meeting_register.to_csv(manifest_path, index=False)
    print("✅ All reference annotations reviewed.")

In [ ]:
from getpass import getpass
from google import genai

api_key = getpass("Enter your Gemini API key: ").strip()

if not api_key:
    raise ValueError("API key is empty.")

client = genai.Client(api_key=api_key)

available_models = [
    model.name
    for model in client.models.list()
    if "generateContent" in (model.supported_actions or [])
]

print("✅ API connected.")
for model_name in available_models:
    print(model_name)

In [ ]:
from google.genai import types

CANDIDATE_MODEL = "gemini-3.5-flash-lite"

try:
    response = client.models.generate_content(
        model=CANDIDATE_MODEL,
        contents='Return exactly this JSON: {"status":"ok"}',
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
        )
    )

    result = json.loads(response.text or "")
    if result.get("status") != "ok":
        raise ValueError("Unexpected test response.")

    MODEL_NAME = CANDIDATE_MODEL
    print("✅ Generation test passed.")
    print("Selected model:", MODEL_NAME)

except Exception as error:
    print("❌ Generation test failed:")
    print(error)

In [ ]:
from pydantic import BaseModel, ConfigDict

class Decision(BaseModel):
    model_config = ConfigDict(extra="forbid")
    task: str
    evidence: str

class ActionItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    task: str
    owner: str
    deadline: str
    condition: str
    evidence: str

class MeetingOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    meeting_title: str
    summary: str
    decisions: list[Decision]
    action_items: list[ActionItem]

def make_api_schema(value):
    if isinstance(value, dict):
        return {
            key: make_api_schema(item)
            for key, item in value.items()
            if key != "additionalProperties"
        }
    if isinstance(value, list):
        return [make_api_schema(item) for item in value]
    return value

API_SCHEMA = make_api_schema(MeetingOutput.model_json_schema())
TEMPERATURE = 0.2

experiment_settings = {
    "model": MODEL_NAME,
    "temperature": TEMPERATURE,
    "runs_per_prompt": 1,
    "prompt_conditions": ["generic", "structured"],
    "input": "human-reviewed transcript",
    "annotation_rules_version": "1.0",
    "distinct_conversations": 13,
    "input_files": 16,
    "annotation_method": "AI-assisted, human-reviewed"
}

(PROJECT_ROOT / "experiment_settings.json").write_text(
    json.dumps(experiment_settings, indent=2),
    encoding="utf-8"
)

(PROJECT_ROOT / "output_schema.json").write_text(
    json.dumps(MeetingOutput.model_json_schema(), indent=2),
    encoding="utf-8"
)

print("✅ Schema and experiment settings saved.")
print("Model:", MODEL_NAME)

In [ ]:
PROMPTS = {
    "generic": (
        "Summarize this meeting and extract its agreed decisions "
        "and future action items. Include supporting transcript quotes. "
        "Use empty strings for unspecified fields."
    ),

    "structured": """
Use only the supplied meeting transcript.

1. Summarize the main discussion.
2. Extract accepted choices or manager instructions as decisions.
   Exclude suggestions without clear acceptance.
3. Extract future assignments and accepted commitments as actions.
   Exclude completed work and general aspirations.
4. Separate independently executable tasks. Keep a report and its
   required contents together. Separate preparing and sharing it.
5. Record each distinct task once.
6. Use explicitly named owners. For ambiguous 'we' commitments,
   record the speaker without inventing a team name. If no owner
   can be identified, use an empty string.
7. Preserve deadline wording. Do not transfer a deadline from one
   task to another. Leave unspecified deadlines empty.
8. Record conditional triggers separately from deadlines.
9. Include exact supporting quotes, including acceptance where needed.
10. Do not invent tasks, owners, dates or decisions.
"""
}

(PROJECT_ROOT / "prompts.json").write_text(
    json.dumps(PROMPTS, indent=2),
    encoding="utf-8"
)

print("✅ Both prompts saved.")

In [ ]:
import hashlib

def run_extraction_batch():
    for index, row in meeting_register.iterrows():
        mid = row["meeting_id"]
        cid = row["conversation_id"]

        annotation = json.loads(
            (DIRS["annotations"] / f"{cid}_reference_reviewed.json")
            .read_text(encoding="utf-8")
        )
        if annotation.get("status") != "human_reviewed":
            raise ValueError(f"{cid}: reference review missing.")

        text = (
            DIRS["reviewed_transcripts"]
            / row["reviewed_transcript_filename"]
        ).read_text(encoding="utf-8")

        for prompt_type, prompt in PROMPTS.items():
            output_path = (
                DIRS[f"{prompt_type}_outputs"]
                / f"{mid}_{prompt_type}_run1.json"
            )

            if output_path.exists():
                MeetingOutput.model_validate_json(
                    output_path.read_text(encoding="utf-8")
                )
                meeting_register.loc[
                    index, f"{prompt_type}_status"
                ] = "success"
                meeting_register.to_csv(manifest_path, index=False)
                print(f"✅ {mid} {prompt_type}: saved output reused.")
                continue

            stamp = datetime.now(timezone.utc).strftime(
                "%Y%m%d_%H%M%S_%f"
            )
            log_path = (
                DIRS["logs"] / f"{mid}_{prompt_type}_{stamp}.json"
            )

            log = {
                "meeting_id": mid,
                "conversation_id": cid,
                "input_type": row["input_type"],
                "model": MODEL_NAME,
                "temperature": TEMPERATURE,
                "prompt_type": prompt_type,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "transcript_sha256": hashlib.sha256(
                    text.encode("utf-8")
                ).hexdigest(),
            }
            started = time.perf_counter()

            try:
                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=(
                        prompt + "\n\nMEETING TRANSCRIPT:\n" + text
                    ),
                    config=types.GenerateContentConfig(
                        temperature=TEMPERATURE,
                        response_mime_type="application/json",
                        response_schema=API_SCHEMA,
                    )
                )

                log_path.with_suffix(".raw.txt").write_text(
                    response.text or "", encoding="utf-8"
                )

                validated = MeetingOutput.model_validate_json(
                    response.text or ""
                )
                output_path.write_text(
                    validated.model_dump_json(indent=2),
                    encoding="utf-8"
                )

                log["status"] = "success"
                meeting_register.loc[
                    index, f"{prompt_type}_status"
                ] = "success"

                print(
                    f"✅ {mid} {prompt_type}: "
                    f"{len(validated.decisions)} decisions, "
                    f"{len(validated.action_items)} actions."
                )

            except Exception as error:
                log["status"] = "failed"
                log["error"] = str(error)
                meeting_register.loc[
                    index, f"{prompt_type}_status"
                ] = "failed"
                print(f"❌ {mid} {prompt_type}: {error}")

            finally:
                log["processing_seconds"] = round(
                    time.perf_counter() - started, 3
                )
                log_path.write_text(
                    json.dumps(log, indent=2), encoding="utf-8"
                )
                meeting_register.to_csv(manifest_path, index=False)

            if log["status"] == "failed":
                print("Batch stopped. Successful outputs preserved.")
                return

    print("✅ All 32 outputs saved.")

run_extraction_batch()

In [ ]:
from google.colab import files
import shutil

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

backup_path = shutil.make_archive(
    f"/content/AIMIS_Text_Audio_Backup_{stamp}",
    "zip",
    root_dir=str(PROJECT_ROOT.parent),
    base_dir=PROJECT_ROOT.name
)

files.download(backup_path)
print("✅ Backup includes all 32 outputs and reviewed annotations.")

In [ ]:
normalization_choices = [
    (f"{mid} — {prompt}", (mid, prompt))
    for mid in meeting_register["meeting_id"]
    for prompt in ["generic", "structured"]
]

norm_selector = widgets.Dropdown(
    options=normalization_choices,
    description="Output:"
)
norm_editor = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="450px")
)
norm_checked = widgets.Checkbox(
    description="I checked task splitting and duplicates.",
    value=False
)
norm_save = widgets.Button(description="Save task units")
norm_message = widgets.Output()

def load_task_units(change=None):
    mid, prompt = norm_selector.value
    original_path = (
        DIRS[f"{prompt}_outputs"] / f"{mid}_{prompt}_run1.json"
    )
    original = json.loads(
        original_path.read_text(encoding="utf-8")
    )

    saved_path = (
        DIRS["evaluation_evidence"] / f"{mid}_{prompt}_task_units.json"
    )

    if saved_path.exists():
        data = json.loads(saved_path.read_text(encoding="utf-8"))
    else:
        data = {
            "meeting_id": mid,
            "prompt": prompt,
            "status": "pending_review",
            "action_items": [
                {
                    **item,
                    "source_action": i,
                    "exclude_duplicate": False,
                    "duplicate_reason": ""
                }
                for i, item in enumerate(original["action_items"], 1)
            ]
        }

    norm_editor.value = json.dumps(data, indent=2, ensure_ascii=False)
    norm_checked.value = False
    norm_message.clear_output()

def save_task_units(_):
    with norm_message:
        norm_message.clear_output()
        try:
            if not norm_checked.value:
                raise ValueError("Tick the review box first.")

            mid, prompt = norm_selector.value
            data = json.loads(norm_editor.value)

            if data["meeting_id"] != mid or data["prompt"] != prompt:
                raise ValueError("Keep the selected meeting and prompt IDs.")

            original = json.loads(
                (DIRS[f"{prompt}_outputs"]
                 / f"{mid}_{prompt}_run1.json")
                .read_text(encoding="utf-8")
            )

            represented = set()

            for i, item in enumerate(data["action_items"], 1):
                source_id = item["source_action"]
                if type(source_id) is not int:
                    raise ValueError("source_action must be an integer.")
                if not 1 <= source_id <= len(original["action_items"]):
                    raise ValueError("Invalid source_action.")

                source_item = original["action_items"][source_id - 1]
                represented.add(source_id)

                # Preserve generated fields, including their errors.
                for field in ["owner", "deadline", "condition", "evidence"]:
                    if item[field] != source_item[field]:
                        raise ValueError(
                            f"Preserve original {field} for source {source_id}."
                        )

                if not item["task"].strip():
                    raise ValueError("Task cannot be empty.")
                if type(item["exclude_duplicate"]) is not bool:
                    raise ValueError("exclude_duplicate must be true or false.")
                if item["exclude_duplicate"] and not item["duplicate_reason"].strip():
                    raise ValueError("Explain each excluded duplicate.")

                item["evaluation_id"] = f"A{i}"

            if represented != set(range(1, len(original["action_items"]) + 1)):
                raise ValueError("Keep a record of every original action.")

            data["status"] = "human_reviewed"
            data["reviewed_at"] = datetime.now(timezone.utc).isoformat()
            data["reviewer"] = "wajahat"

            (
                DIRS["evaluation_evidence"]
                / f"{mid}_{prompt}_task_units.json"
            ).write_text(
                json.dumps(data, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )
            print(f"✅ {mid} {prompt}: task units saved.")

        except Exception as error:
            print("Not saved:", error)

norm_selector.observe(load_task_units, names="value")
norm_save.on_click(save_task_units)

display(norm_selector, norm_editor, norm_checked, norm_save, norm_message)
load_task_units()

In [ ]:
pending = []

for mid in meeting_register["meeting_id"]:
    for prompt in ["generic", "structured"]:
        path = (
            DIRS["evaluation_evidence"]
            / f"{mid}_{prompt}_task_units.json"
        )

        if not path.exists():
            pending.append(f"{mid} {prompt}")
        else:
            data = json.loads(path.read_text(encoding="utf-8"))
            if data.get("status") != "human_reviewed":
                pending.append(f"{mid} {prompt}")

print("Saved reviews:", 32 - len(pending), "/ 32")
print("Pending:", ", ".join(pending) if pending else "None")

In [ ]:
import re

flagged_rows = []

for mid in meeting_register["meeting_id"]:
    for prompt in ["generic", "structured"]:
        path = (
            DIRS["evaluation_evidence"]
            / f"{mid}_{prompt}_task_units.json"
        )
        data = json.loads(path.read_text(encoding="utf-8"))

        for item in data["action_items"]:
            if item["exclude_duplicate"]:
                continue

            if re.search(
                r"\b(and|while|ensure)\b",
                item["task"],
                flags=re.IGNORECASE
            ):
                flagged_rows.append({
                    "meeting_id": mid,
                    "prompt": prompt,
                    "evaluation_id": item["evaluation_id"],
                    "source_action": item["source_action"],
                    "task": item["task"]
                })

flagged_tasks = pd.DataFrame(flagged_rows)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(flagged_tasks)
print("Flags are review aids; some tasks correctly belong together.")

In [ ]:
import copy

split_rules = {}

def add_split(mid, prompt, source, tasks):
    split_rules[(mid, prompt, source)] = tasks

# Original progress meeting
for mid in ["M01", "M03", "M04"]:
    add_split(mid, "generic", 2, [
        "Continue cleaning existing data.",
        "Continue analyzing existing data.",
        "Flag incomplete responses."
    ])

for mid in ["M01", "M04"]:
    add_split(mid, "generic", 3, [
        "Continue working on the report.",
        "Ensure consistent use of employee participation."
    ])

# Project development meeting
add_split("M02", "generic", 3, [
    "Complete remaining technical improvements.",
    "Resolve identified issues."
])
add_split("M02", "generic", 4, [
    "Continue the quality review.",
    "Update documentation requirements."
])
add_split("M02", "structured", 8, [
    "Coordinate with all departments.",
    "Distribute the updated plan."
])
add_split("M02", "structured", 9, [
    "Complete remaining technical improvements.",
    "Resolve identified issues."
])
add_split("M02", "structured", 11, [
    "Continue the quality review.",
    "Update documentation requirements."
])
add_split("M02", "structured", 12, [
    "Complete the quality review.",
    "Share quality-review observations."
])

add_split("M05", "generic", 1, [
    "Complete remaining technical improvements.",
    "Resolve identified issues."
])
add_split("M05", "generic", 2, [
    "Continue the quality review.",
    "Update documentation requirements."
])
add_split("M05", "generic", 3, [
    "Coordinate testing support.",
    "Arrange user feedback sessions.",
    "Update the project timeline."
])

add_split("M05", "structured", 1, [
    "Contact the testing department.",
    "Prepare a testing schedule."
])
add_split("M05", "structured", 4, [
    "Prepare the feedback-session user list.",
    "Share the feedback-session schedule."
])
add_split("M05", "structured", 5, [
    "Complete remaining technical improvements.",
    "Resolve identified issues."
])
add_split("M05", "structured", 6, [
    "Continue the quality review.",
    "Update documentation requirements."
])
add_split("M05", "structured", 7, [
    "Coordinate testing support.",
    "Arrange user feedback sessions.",
    "Update the project timeline."
])
add_split("M05", "structured", 9, [
    "Complete the quality review.",
    "Share quality-review observations."
])
add_split("M05", "structured", 10, [
    "Coordinate with all departments.",
    "Distribute the updated plan."
])

# Employee training
add_split("M06", "generic", 1, [
    "Prepare the detailed training plan with objectives, schedule, cost and outcomes.",
    "Share the first training-plan draft."
])
add_split("M06", "generic", 3, [
    "Collect employee feedback.",
    "Share employee suggestions with the training team."
])
add_split("M06", "structured", 4, [
    "Collect employee feedback.",
    "Share employee suggestions with the training team."
])

# Marketing
for prompt in ["generic", "structured"]:
    add_split("M07", prompt, 3, [
        "Organize the latest customer feedback.",
        "Prepare a summary of customer concerns and suggestions."
    ])

# Project planning
add_split("M08", "generic", 1, [
    "Prepare the tracking system.",
    "Share the tracking system."
])
add_split("M08", "generic", 2, [
    "Coordinate with all departments.",
    "Distribute the updated schedule."
])
add_split("M08", "generic", 3, [
    "Start assigned tasks.",
    "Provide weekly progress updates."
])
add_split("M08", "structured", 3, [
    "Coordinate with all departments.",
    "Distribute the updated schedule."
])
add_split("M08", "structured", 4, [
    "Start assigned tasks.",
    "Provide weekly progress updates."
])

# Budget review
add_split("M09", "generic", 1, [
    "Prepare the financial summary.",
    "Share the financial summary."
])
add_split("M09", "generic", 2, [
    "Review departmental requirements.",
    "Prioritize important expenses."
])
add_split("M09", "structured", 4, [
    "Review departmental requirements.",
    "Prioritize important expenses."
])

# Recruitment
add_split("M11", "generic", 1, [
    "Prepare a candidate shortlist.",
    "Arrange interviews."
])

# Training evaluation
for prompt in ["generic", "structured"]:
    add_split("M12", prompt, 2, [
        "Collect participant feedback.",
        "Prepare the final evaluation report."
    ])

# IT upgrade
add_split("M13", "generic", 1, [
    "Prepare the implementation schedule.",
    "Identify implementation risks."
])
add_split("M13", "generic", 2, [
    "Coordinate with all departments.",
    "Share the final plan."
])
add_split("M13", "structured", 1, [
    "Arrange employee training sessions.",
    "Provide user guides."
])
add_split("M13", "structured", 2, [
    "Prepare the implementation schedule.",
    "Identify implementation risks."
])

# Safety
for prompt in ["generic", "structured"]:
    add_split("M15", prompt, 1, [
        "Prepare a safety training schedule.",
        "Prepare a safety checklist."
    ])

for mid in meeting_register["meeting_id"]:
    for prompt in ["generic", "structured"]:
        path = (
            DIRS["evaluation_evidence"]
            / f"{mid}_{prompt}_task_units.json"
        )
        saved = json.loads(path.read_text(encoding="utf-8"))

        # Preserve the earlier review.
        previous = path.with_name(
            f"{mid}_{prompt}_task_units_before_split.json"
        )
        if not previous.exists():
            previous.write_text(
                json.dumps(saved, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )

        original = json.loads(
            (DIRS[f"{prompt}_outputs"] / f"{mid}_{prompt}_run1.json")
            .read_text(encoding="utf-8")
        )

        proposed = []
        for source, item in enumerate(original["action_items"], 1):
            tasks = split_rules.get(
                (mid, prompt, source), [item["task"]]
            )
            for task in tasks:
                proposed.append({
                    **copy.deepcopy(item),
                    "task": task,
                    "source_action": source,
                    "original_task": item["task"],
                    "evaluation_id": f"A{len(proposed) + 1}",
                    "exclude_duplicate": False,
                    "duplicate_reason": ""
                })

        data = {
            "meeting_id": mid,
            "prompt": prompt,
            "status": "proposed_splits_pending_review",
            "action_items": proposed
        }
        path.write_text(
            json.dumps(data, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )
        print(f"{mid} {prompt}: {len(proposed)} task units")

print("✅ Proposed splits saved. Fields and original outputs preserved.")

In [ ]:
for mid in ["M02", "M05"]:
    path = (
        DIRS["evaluation_evidence"]
        / f"{mid}_structured_task_units.json"
    )
    data = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n===== {mid} STRUCTURED =====")
    for item in data["action_items"]:
        print(
            f"{item['evaluation_id']} "
            f"(source {item['source_action']}): {item['task']}"
        )

In [ ]:
duplicate_rules = {
    "M02": {
        "A9": "Repeats A3: distributing the updated project plan.",
        "A13": "Same quality-review task as A15; retain completion wording."
    },
    "M05": {
        "A1": "Contacting testing is part of A11: coordinating testing support.",
        "A9": "Same quality-review task as A15; retain completion wording.",
        "A12": "Repeats A4: arranging the user feedback session.",
        "A18": "Repeats A3: distributing the updated project plan."
    }
}

for mid, rules in duplicate_rules.items():
    path = (
        DIRS["evaluation_evidence"]
        / f"{mid}_structured_task_units.json"
    )
    data = json.loads(path.read_text(encoding="utf-8"))

    for item in data["action_items"]:
        reason = rules.get(item["evaluation_id"])
        if reason:
            item["exclude_duplicate"] = True
            item["duplicate_reason"] = reason

    data["status"] = "proposed_splits_and_duplicates_pending_review"

    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    retained = sum(
        not item["exclude_duplicate"]
        for item in data["action_items"]
    )
    print(f"✅ {mid} structured: {retained} retained task units.")

print("Duplicate records preserved for review.")

In [ ]:
load_task_units()

display(
    norm_selector,
    norm_editor,
    norm_checked,
    norm_save,
    norm_message
)

In [ ]:
pending = []

for mid in meeting_register["meeting_id"]:
    for prompt in ["generic", "structured"]:
        path = (
            DIRS["evaluation_evidence"]
            / f"{mid}_{prompt}_task_units.json"
        )
        data = json.loads(path.read_text(encoding="utf-8"))

        if data.get("status") != "human_reviewed":
            pending.append(f"{mid} {prompt}")

print("Reviewed:", 32 - len(pending), "/ 32")
print("Pending:", ", ".join(pending) if pending else "None")

In [ ]:
from rapidfuzz.fuzz import ratio

def propose_matches(predicted, reference):
    candidates = []

    for prediction in predicted:
        for target in reference:
            similarity = ratio(
                prediction["task"].lower(),
                target["task"].lower()
            )
            candidates.append(
                (similarity, prediction["id"], target["id"])
            )

    used_outputs = set()
    used_references = set()
    selected = {}

    for similarity, output_id, reference_id in sorted(
        candidates, reverse=True
    ):
        if similarity < 55:
            continue
        if output_id in used_outputs or reference_id in used_references:
            continue

        selected[output_id] = (reference_id, round(similarity, 2))
        used_outputs.add(output_id)
        used_references.add(reference_id)

    return [
        {
            "output_id": prediction["id"],
            "output_task": prediction["task"],
            "reference_id": selected.get(
                prediction["id"], ("", None)
            )[0],
            "similarity_percent": selected.get(
                prediction["id"], ("", None)
            )[1],
            "match_judgment": "pending",
            "reason": ""
        }
        for prediction in predicted
    ]

for _, row in meeting_register.iterrows():
    mid = row["meeting_id"]
    cid = row["conversation_id"]

    reference = json.loads(
        (DIRS["annotations"] / f"{cid}_reference_reviewed.json")
        .read_text(encoding="utf-8")
    )

    for prompt in ["generic", "structured"]:
        match_path = (
            DIRS["evaluation_evidence"]
            / f"{mid}_{prompt}_matches.json"
        )
        if match_path.exists():
            print(f"{mid} {prompt}: existing matches preserved.")
            continue

        output = json.loads(
            (DIRS[f"{prompt}_outputs"] / f"{mid}_{prompt}_run1.json")
            .read_text(encoding="utf-8")
        )
        units = json.loads(
            (DIRS["evaluation_evidence"]
             / f"{mid}_{prompt}_task_units.json")
            .read_text(encoding="utf-8")
        )

        if units["status"] != "human_reviewed":
            raise ValueError(f"{mid} {prompt}: task-unit review missing.")

        decisions = [
            {"id": f"D{i}", "task": item["task"]}
            for i, item in enumerate(output["decisions"], 1)
        ]
        actions = [
            {"id": item["evaluation_id"], "task": item["task"]}
            for item in units["action_items"]
            if not item["exclude_duplicate"]
        ]

        proposed = {
            "meeting_id": mid,
            "conversation_id": cid,
            "prompt": prompt,
            "status": "proposed_pending_review",
            "decisions": propose_matches(
                decisions, reference["decisions"]
            ),
            "action_items": propose_matches(
                actions, reference["action_items"]
            )
        }

        match_path.write_text(
            json.dumps(proposed, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )
        print(f"✅ {mid} {prompt}: proposed matches saved.")

print("No final scores calculated yet.")

In [ ]:
match_selector = widgets.Dropdown(
    options=normalization_choices,
    description="Output:"
)
match_reference = widgets.Output()
match_editor = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="450px")
)
match_reviewer = widgets.Text(value="wajahat", description="Reviewer:")
match_checked = widgets.Checkbox(
    description="I checked every task match.", value=False
)
match_save = widgets.Button(description="Save reviewed matches")
match_message = widgets.Output()

def load_matches(change=None):
    mid, prompt = match_selector.value
    path = DIRS["evaluation_evidence"] / f"{mid}_{prompt}_matches.json"
    data = json.loads(path.read_text(encoding="utf-8"))

    match_editor.value = json.dumps(data, indent=2, ensure_ascii=False)
    match_checked.value = False
    match_message.clear_output()

    reference = json.loads(
        (DIRS["annotations"]
         / f"{data['conversation_id']}_reference_reviewed.json")
        .read_text(encoding="utf-8")
    )

    with match_reference:
        match_reference.clear_output()
        for key in ["decisions", "action_items"]:
            print("\nREFERENCE", key.upper())
            for item in reference[key]:
                print(item["id"], "—", item["task"])

def save_matches(_):
    with match_message:
        match_message.clear_output()
        try:
            if not match_checked.value or not match_reviewer.value.strip():
                raise ValueError("Enter reviewer and tick the review box.")

            mid, prompt = match_selector.value
            path = DIRS["evaluation_evidence"] / f"{mid}_{prompt}_matches.json"
            original = json.loads(path.read_text(encoding="utf-8"))
            data = json.loads(match_editor.value)

            for field in ["meeting_id", "conversation_id", "prompt"]:
                if data[field] != original[field]:
                    raise ValueError(f"Keep {field} unchanged.")

            reference = json.loads(
                (DIRS["annotations"]
                 / f"{data['conversation_id']}_reference_reviewed.json")
                .read_text(encoding="utf-8")
            )

            for key in ["decisions", "action_items"]:
                original_items = {
                    item["output_id"]: item for item in original[key]
                }
                items = data[key]
                ids = [item["output_id"] for item in items]

                if len(ids) != len(set(ids)) or set(ids) != set(original_items):
                    raise ValueError("Keep every output exactly once.")

                valid_references = {item["id"] for item in reference[key]}
                assigned = []

                for item in items:
                    if item["output_task"] != original_items[
                        item["output_id"]
                    ]["output_task"]:
                        raise ValueError("Keep output_task unchanged.")

                    judgment = item["match_judgment"]
                    if judgment not in ["full", "partial", "unmatched"]:
                        raise ValueError("Replace every pending judgment.")

                    if judgment == "unmatched":
                        if item["reference_id"] != "":
                            raise ValueError("Unmatched items need an empty reference_id.")
                    else:
                        if item["reference_id"] not in valid_references:
                            raise ValueError("Invalid reference_id.")
                        assigned.append(item["reference_id"])

                    if not item["reason"].strip():
                        raise ValueError("Give a short reason for each judgment.")

                if len(assigned) != len(set(assigned)):
                    raise ValueError("Each reference can be assigned only once.")

            data["status"] = "human_reviewed"
            data["reviewer"] = match_reviewer.value.strip()
            data["reviewed_at"] = datetime.now(timezone.utc).isoformat()

            path.write_text(
                json.dumps(data, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )
            print(f"✅ {mid} {prompt}: matches reviewed.")

        except Exception as error:
            print("Not saved:", error)

match_selector.observe(load_matches, names="value")
match_save.on_click(save_matches)

display(
    match_selector, match_reference, match_editor,
    match_reviewer, match_checked, match_save, match_message
)
load_matches()

In [ ]:
path = (
    DIRS["evaluation_evidence"]
    / "M01_generic_matches.json"
)

print(path.read_text(encoding="utf-8"))

In [ ]:
path = DIRS["evaluation_evidence"] / "M01_generic_matches.json"
data = json.loads(path.read_text(encoding="utf-8"))

judgments = {
    "decisions": {
        "D1": ("D2", "Same terminology decision."),
        "D2": ("D3", "Same conditional proceed-and-document decision.")
    },
    "action_items": {
        "A1": ("A1", "Same conditional reminder task."),
        "A2": ("A2", "Same data-cleaning task."),
        "A3": ("A3", "Same data-analysis task."),
        "A4": ("A4", "Same incomplete-response flagging task."),
        "A5": ("A5", "Same report-continuation task."),
        "A6": ("A6", "Same report terminology-standardization task."),
        "A7": ("A7", "Same updated-draft sending task; recipient omitted.")
    }
}

for key, mapping in judgments.items():
    for item in data[key]:
        reference_id, reason = mapping[item["output_id"]]
        item["reference_id"] = reference_id
        item["match_judgment"] = "full"
        item["reason"] = reason

data["status"] = "AI_assisted_judgments_pending_human_review"

path.write_text(
    json.dumps(data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("✅ Proposed judgments saved.")
print("Missing reference decision: D1")
print("Missing reference action: A8")

In [ ]:
confirmed = input(
    "Do you agree with the M01 generic task matches? yes/no: "
).strip().lower()

if confirmed == "yes":
    path = DIRS["evaluation_evidence"] / "M01_generic_matches.json"
    data = json.loads(path.read_text(encoding="utf-8"))

    data["status"] = "human_reviewed"
    data["reviewer"] = "wajahat"
    data["reviewed_at"] = datetime.now(timezone.utc).isoformat()

    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("✅ M01 generic task-match review saved.")
else:
    print("Review pending. Tell me which match needs changing.")

In [ ]:
path = (
    DIRS["evaluation_evidence"]
    / "M01_structured_matches.json"
)

print(path.read_text(encoding="utf-8"))

print("\nORIGINAL STRUCTURED OUTPUT:")
output_path = (
    DIRS["structured_outputs"] / "M01_structured_run1.json"
)
print(output_path.read_text(encoding="utf-8"))

In [ ]:
row = meeting_register[
    meeting_register["meeting_id"] == "M01"
].iloc[0]

path = (
    DIRS["reviewed_transcripts"]
    / row["reviewed_transcript_filename"]
)

print(path.read_text(encoding="utf-8"))

In [ ]:
corrections = {
    "Perfect. Asia,": "Perfect. Aisha,",
    "Asha will continue": "Aisha will continue",
    "fold the entire project": "hold the entire project",
    "renew the report": "review the report",
    "renew everything": "review everything",
    "a updated draft": "an updated draft",
}

original_text = path.read_text(encoding="utf-8")
corrected_text = original_text

for old, new in corrections.items():
    corrected_text = corrected_text.replace(old, new)

for old, new in corrections.items():
    print(f"{old} → {new}")

confirmed = input(
    "Have you checked these corrections against the audio? yes/no: "
).strip().lower()

if confirmed == "yes":
    previous = path.with_name("M01_reviewed_before_correction.txt")
    if not previous.exists():
        previous.write_text(original_text, encoding="utf-8")

    path.write_text(corrected_text, encoding="utf-8")

    correction_log = {
        "meeting_id": "M01",
        "reviewer": "wajahat",
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
        "corrections": corrections,
        "reason": "Errors remained in the first reviewed transcript"
    }
    (DIRS["logs"] / "M01_transcript_corrections.json").write_text(
        json.dumps(correction_log, indent=2), encoding="utf-8"
    )

    print("✅ Corrected M01 saved. Raw Whisper text preserved.")
else:
    print("Nothing changed.")

In [ ]:
row = meeting_register[
    meeting_register["meeting_id"] == "M01"
].iloc[0]

raw_text = (
    DIRS["raw_transcripts"] / row["raw_transcript_filename"]
).read_text(encoding="utf-8")

reviewed_text = (
    DIRS["reviewed_transcripts"]
    / row["reviewed_transcript_filename"]
).read_text(encoding="utf-8")

reference = normalize_asr(reviewed_text)
hypothesis = normalize_asr(raw_text)
comparison = process_words(reference, hypothesis)

metrics_path = DIRS["results"] / "transcription_metrics.csv"
asr_results = pd.read_csv(metrics_path)

updates = {
    "reference_words": len(reference.split()),
    "substitutions": comparison.substitutions,
    "deletions": comparison.deletions,
    "insertions": comparison.insertions,
    "WER_percent": round(100 * comparison.wer, 2),
    "CER_percent": round(100 * cer(reference, hypothesis), 2),
    "reference_method": "human-corrected Whisper transcript"
}

for field, value in updates.items():
    asr_results.loc[
        asr_results["meeting_id"] == "M01", field
    ] = value

asr_results.to_csv(metrics_path, index=False)

display(asr_results[asr_results["meeting_id"] == "M01"])
print("✅ M01 transcription score updated.")

In [ ]:
import shutil
import hashlib

archive_dir = (
    DIRS["evaluation_evidence"] / "M01_before_transcript_correction"
)
archive_dir.mkdir(parents=True, exist_ok=True)

for prompt in ["generic", "structured"]:
    old_output = (
        DIRS[f"{prompt}_outputs"] / f"M01_{prompt}_run1.json"
    )
    if old_output.exists():
        archived = archive_dir / old_output.name
        if not archived.exists():
            shutil.copy2(old_output, archived)

    for suffix in ["task_units", "matches"]:
        old_evidence = (
            DIRS["evaluation_evidence"]
            / f"M01_{prompt}_{suffix}.json"
        )
        if old_evidence.exists():
            archived = archive_dir / old_evidence.name
            if not archived.exists():
                shutil.copy2(old_evidence, archived)

row = meeting_register[
    meeting_register["meeting_id"] == "M01"
].iloc[0]

text = (
    DIRS["reviewed_transcripts"]
    / row["reviewed_transcript_filename"]
).read_text(encoding="utf-8")

for prompt, instruction in PROMPTS.items():
    new_path = (
        DIRS[f"{prompt}_outputs"] / f"M01_{prompt}_corrected_run1.json"
    )

    if new_path.exists():
        print(f"✅ {prompt}: corrected output already saved.")
        continue

    started = time.perf_counter()

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=instruction + "\n\nMEETING TRANSCRIPT:\n" + text,
            config=types.GenerateContentConfig(
                temperature=TEMPERATURE,
                response_mime_type="application/json",
                response_schema=API_SCHEMA
            )
        )

        (DIRS["logs"] / f"M01_{prompt}_corrected_raw.txt").write_text(
            response.text or "", encoding="utf-8"
        )

        validated = MeetingOutput.model_validate_json(
            response.text or ""
        )
        new_path.write_text(
            validated.model_dump_json(indent=2), encoding="utf-8"
        )

        log = {
            "meeting_id": "M01",
            "prompt": prompt,
            "model": MODEL_NAME,
            "temperature": TEMPERATURE,
            "transcript_sha256": hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest(),
            "processing_seconds": round(
                time.perf_counter() - started, 3
            ),
            "status": "success"
        }
        (DIRS["logs"] / f"M01_{prompt}_corrected_log.json").write_text(
            json.dumps(log, indent=2), encoding="utf-8"
        )

        print(
            f"✅ {prompt}: {len(validated.decisions)} decisions, "
            f"{len(validated.action_items)} actions."
        )

    except Exception as error:
        print(f"❌ {prompt}: {error}")
        break


In [ ]:
for prompt in ["generic", "structured"]:
    path = (
        DIRS[f"{prompt}_outputs"]
        / f"M01_{prompt}_corrected_run1.json"
    )

    print(f"\n===== M01 CORRECTED {prompt.upper()} =====")
    print(path.read_text(encoding="utf-8"))

In [ ]:
corrected_mappings = {
    "generic": {
        "decisions": [
            ("D1", "D2", "full", "Same terminology decision."),
            ("D2", "D3", "full", "Same conditional proceed-and-document decision.")
        ],
        "action_items": [
            ("A1", "A1", "full", "Same reminder task; timing scored separately."),
            ("A2", "A4", "full", "Same incomplete-response flagging task."),
            ("A3", "A6", "full", "Same terminology-standardization task."),
            ("A4", "A7", "full", "Same draft-sending task; recipient omitted."),
            ("A5", "A8", "full", "Same report-review task; owner scored separately.")
        ]
    },
    "structured": {
        "decisions": [
            ("D1", "D2", "full", "Same terminology decision.")
        ],
        "action_items": [
            ("A1", "A1", "full", "Same reminder task; condition scored separately."),
            ("A2", "A4", "full", "Same incomplete-response flagging task."),
            ("A3", "A6", "full", "Same terminology-standardization task."),
            ("A4", "", "unmatched",
             "Reference treats this as decision D3, not a separately assigned action."),
            ("A5", "A8", "full", "Same report-review task; owner scored separately."),
            ("A6", "A7", "full", "Same draft-sending task; recipient omitted.")
        ]
    }
}

for prompt, mapping in corrected_mappings.items():
    output_path = (
        DIRS[f"{prompt}_outputs"]
        / f"M01_{prompt}_corrected_run1.json"
    )
    output = json.loads(output_path.read_text(encoding="utf-8"))

    matches = {
        "meeting_id": "M01",
        "conversation_id": "C01",
        "prompt": prompt,
        "output_filename": output_path.name,
        "status": "AI_assisted_judgments_pending_human_review"
    }

    for key in ["decisions", "action_items"]:
        matches[key] = []
        for output_id, reference_id, judgment, reason in mapping[key]:
            item = output[key][int(output_id[1:]) - 1]
            matches[key].append({
                "output_id": output_id,
                "output_task": item["task"],
                "reference_id": reference_id,
                "match_judgment": judgment,
                "reason": reason
            })

    (
        DIRS["evaluation_evidence"]
        / f"M01_{prompt}_corrected_matches.json"
    ).write_text(
        json.dumps(matches, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"✅ {prompt}: corrected-run proposed matches saved.")

In [ ]:
confirmed = input(
    "Do you agree with both corrected M01 task-match sets? yes/no: "
).strip().lower()

if confirmed == "yes":
    reference = json.loads(
        (DIRS["annotations"] / "C01_reference_reviewed.json")
        .read_text(encoding="utf-8")
    )
    metric_rows = []

    for prompt in ["generic", "structured"]:
        match_path = (
            DIRS["evaluation_evidence"]
            / f"M01_{prompt}_corrected_matches.json"
        )
        matches = json.loads(match_path.read_text(encoding="utf-8"))
        output = json.loads(
            (DIRS[f"{prompt}_outputs"] / matches["output_filename"])
            .read_text(encoding="utf-8")
        )

        for key, label in [
            ("decisions", "decision"),
            ("action_items", "action")
        ]:
            full = [
                item for item in matches[key]
                if item["match_judgment"] == "full"
            ]
            reference_ids = [item["reference_id"] for item in full]
            if len(reference_ids) != len(set(reference_ids)):
                raise ValueError("Duplicate reference matches.")

            tp = len(full)
            fp = len(output[key]) - tp
            fn = len(reference[key]) - tp

            metric_rows.append({
                "meeting_id": "M01",
                "run": "corrected_run1",
                "prompt": prompt,
                "item_type": label,
                "TP": tp,
                "FP_reference_unmatched": fp,
                "FN": fn,
                "precision_percent": (
                    round(100 * tp / (tp + fp), 2)
                    if tp + fp else None
                ),
                "recall_percent": (
                    round(100 * tp / (tp + fn), 2)
                    if tp + fn else None
                ),
                "f1_percent": (
                    round(200 * tp / (2 * tp + fp + fn), 2)
                    if 2 * tp + fp + fn else None
                ),
                "status": "human_reviewed"
            })

        matches["status"] = "human_reviewed"
        matches["reviewer"] = "wajahat"
        matches["reviewed_at"] = datetime.now(timezone.utc).isoformat()
        match_path.write_text(
            json.dumps(matches, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    m01_corrected_metrics = pd.DataFrame(metric_rows)
    m01_corrected_metrics.to_csv(
        DIRS["results"] / "M01_corrected_task_metrics.csv",
        index=False
    )
    display(m01_corrected_metrics)
else:
    print("Scores not finalised. Tell me the required correction.")

In [ ]:
reference = json.loads(
    (DIRS["annotations"] / "C01_reference_reviewed.json")
    .read_text(encoding="utf-8")
)
reference_actions = {
    item["id"]: item for item in reference["action_items"]
}

# Proposed field errors from the corrected outputs.
errors = {
    ("generic", "A1", "deadline"):
        "Tomorrow morning is the reminder trigger, not a completion deadline.",
    ("generic", "A1", "condition"):
        "Condition omits the tomorrow-morning trigger.",
    ("generic", "A2", "condition"):
        "For now is temporal wording, not a conditional trigger.",
    ("generic", "A5", "owner"):
        "Sarah is the reference owner; output owner is blank.",

    ("structured", "A1", "condition"):
        "Condition omits the tomorrow-morning trigger.",
    ("structured", "A2", "condition"):
        "For now is temporal wording, not a conditional trigger.",
    ("structured", "A5", "owner"):
        "Sarah is the reference owner; output owner is blank."
}

field_rows = []

for prompt in ["generic", "structured"]:
    matches = json.loads(
        (DIRS["evaluation_evidence"]
         / f"M01_{prompt}_corrected_matches.json")
        .read_text(encoding="utf-8")
    )
    output = json.loads(
        (DIRS[f"{prompt}_outputs"] / matches["output_filename"])
        .read_text(encoding="utf-8")
    )

    for match in matches["action_items"]:
        if match["match_judgment"] != "full":
            continue

        predicted = output["action_items"][
            int(match["output_id"][1:]) - 1
        ]
        target = reference_actions[match["reference_id"]]

        for field in ["owner", "deadline", "condition"]:
            reason = errors.get((prompt, match["output_id"], field))

            field_rows.append({
                "prompt": prompt,
                "output_id": match["output_id"],
                "reference_id": match["reference_id"],
                "field": field,
                "reference_value": target[field],
                "predicted_value": predicted[field],
                "correct": reason is None,
                "reason": reason or "Equivalent or correctly unspecified.",
                "status": "proposed_pending_review"
            })

m01_field_checks = pd.DataFrame(field_rows)

m01_field_checks.to_csv(
    DIRS["evaluation_evidence"] / "M01_corrected_field_checks.csv",
    index=False
)

display(m01_field_checks)

In [ ]:
confirmed = input(
    "Do you agree with the M01 owner, deadline and condition checks? yes/no: "
).strip().lower()

if confirmed == "yes":
    m01_field_checks["status"] = "human_reviewed"
    m01_field_checks["reviewer"] = "wajahat"
    m01_field_checks["reviewed_at"] = (
        datetime.now(timezone.utc).isoformat()
    )

    m01_field_checks.to_csv(
        DIRS["evaluation_evidence"] / "M01_corrected_field_checks.csv",
        index=False
    )

    m01_field_summary = (
        m01_field_checks.groupby(["prompt", "field"])["correct"]
        .agg(correct_count="sum", matched_actions="count")
        .reset_index()
    )

    m01_field_summary["accuracy_percent"] = (
        100 * m01_field_summary["correct_count"]
        / m01_field_summary["matched_actions"]
    ).round(2)

    m01_field_summary.to_csv(
        DIRS["results"] / "M01_corrected_field_metrics.csv",
        index=False
    )

    display(m01_field_summary)
else:
    print("Checks remain provisional. Tell me the correction.")

In [ ]:
for prompt in ["generic", "structured"]:
    print(f"\n===== M02 {prompt.upper()} MATCHES =====")

    match_path = (
        DIRS["evaluation_evidence"]
        / f"M02_{prompt}_matches.json"
    )
    print(match_path.read_text(encoding="utf-8"))

    print("\nGENERATED ACTION FIELDS:")
    units_path = (
        DIRS["evaluation_evidence"]
        / f"M02_{prompt}_task_units.json"
    )
    units = json.loads(units_path.read_text(encoding="utf-8"))

    for item in units["action_items"]:
        if item["exclude_duplicate"]:
            continue
        print(
            item["evaluation_id"],
            "| owner:", item["owner"],
            "| deadline:", item["deadline"],
            "| condition:", item["condition"]
        )


In [ ]:
m02_judgments = {
    "generic": {
        "decisions": {
            "D1": ("D2", "full", "Same quality-first, unhurried-testing decision.")
        },
        "action_items": {
            "A1": ("A7", "full", "Coordinate additional testing support."),
            "A2": ("A10", "full", "Share the updated project timeline."),
            "A3": ("A1", "full", "Complete technical improvements."),
            "A4": ("A2", "full", "Resolve technical issues."),
            "A5": ("A4", "full", "Same ongoing quality-review task."),
            "A6": ("A5", "full", "Update documentation requirements."),
            "A7": ("A11", "full", "Arrange the user feedback session.")
        }
    },
    "structured": {
        "decisions": {
            "D1": ("D2", "partial",
                   "Includes detailed review but omits the explicit no-rushing requirement."),
            "D2": ("D3", "partial",
                   "Includes the feedback session but omits documenting feedback.")
        },
        "action_items": {
            "A1": ("A7", "full", "Coordinate additional testing support."),
            "A2": ("A8", "full", "Prepare the testing schedule."),
            "A3": ("A10", "full", "Share the updated timeline."),
            "A4": ("A11", "full", "Arrange the feedback session."),
            "A5": ("A12", "full", "Ensure feedback is documented."),
            "A6": ("A13", "full", "Prepare the feedback-session user list."),
            "A7": ("A14", "full", "Share the feedback-session schedule."),
            "A8": ("A15", "full", "Coordinate with all departments."),
            "A10": ("A1", "full", "Complete technical improvements."),
            "A11": ("A2", "full", "Resolve technical issues."),
            "A12": ("A3", "full", "Provide the technical progress update."),
            "A14": ("A5", "full", "Update documentation requirements."),
            "A15": ("A4", "full", "Complete the quality review."),
            "A16": ("A6", "full", "Share quality-review observations.")
        }
    }
}

for prompt, groups in m02_judgments.items():
    path = (
        DIRS["evaluation_evidence"]
        / f"M02_{prompt}_matches.json"
    )
    data = json.loads(path.read_text(encoding="utf-8"))

    for key, mapping in groups.items():
        for item in data[key]:
            ref_id, judgment, reason = mapping[item["output_id"]]
            item["reference_id"] = ref_id
            item["match_judgment"] = judgment
            item["reason"] = reason

    data["status"] = "AI_assisted_judgments_pending_human_review"
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"✅ M02 {prompt}: corrected proposals saved.")

In [ ]:
confirmed = input(
    "Do you agree with the proposed M02 matches, including "
    "the two partial structured decisions? yes/no: "
).strip().lower()

if confirmed == "yes":
    for prompt in ["generic", "structured"]:
        path = (
            DIRS["evaluation_evidence"]
            / f"M02_{prompt}_matches.json"
        )
        data = json.loads(path.read_text(encoding="utf-8"))
        data["status"] = "human_reviewed"
        data["reviewer"] = "wajahat"
        data["reviewed_at"] = datetime.now(timezone.utc).isoformat()

        path.write_text(
            json.dumps(data, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    print("✅ M02 task matches reviewed.")
else:
    print("Review pending. Tell me the correction.")

In [ ]:
import shutil
from google.colab import files
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

backup = shutil.make_archive(
    f"/content/AIMIS_Text_Audio_Backup_{stamp}",
    "zip",
    root_dir=str(PROJECT_ROOT.parent),
    base_dir=PROJECT_ROOT.name
)

files.download(backup)
print("✅ Updated project backup ready.")

In [ ]:
import shutil
from google.colab import files
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

backup = shutil.make_archive(
    f"/content/AIMIS_Text_Audio_Backup_{stamp}",
    "zip",
    root_dir=str(PROJECT_ROOT.parent),
    base_dir=PROJECT_ROOT.name
)

files.download(backup)
print("✅ Updated backup ready.")

In [ ]:
# Reference IDs listed in output order.
# "-" means unmatched. "D1:partial" means partial coverage.

batch_maps = {
    "M03": {
        "generic": ("D2 D3", "A1 A2 A3 A4 A6 A7 A8"),
        "structured": ("D1 D2 D3", "A1 A4 A6 A8 A7")
    },
    "M04": {
        "generic": ("D2 D3", "A1 A2 A3 A4 A5 A6 A7 A8"),
        "structured": ("D2 D3", "A1 A4 A6 A7 A8")
    },
    "M05": {
        "generic": ("D2", "A1 A2 A4 A5 A7 A11 A9"),
        "structured": (
            "D2:partial",
            "A8 A10 A11 A13 A14 A1 A2 A5 A7 A9 A3 A4 A6 A15"
        )
    },
    "M06": {
        "generic": ("- - - - D1 D2", "A1 A2 A3 A4 A5"),
        "structured": ("- - - D1 D2", "A1 A2 A3 A4 A5")
    },
    "M07": {
        "generic": ("D2 -", "A1 A2 A3 A4"),
        "structured": ("- D1:partial - D2", "A1 A2 A3 A4 A5 A6")
    },
    "M08": {
        "generic": ("D1", "A1 A2 A3 A4 A5 A6"),
        "structured": ("D1", "A1 A2 A3 A4 A5 A6")
    },
    "M09": {
        "generic": ("D1:partial", "A2 A3 A4 A5"),
        "structured": ("D1", "A1 A2 A3 A4 A5")
    },
    "M10": {
        "generic": ("D1", "A1 A2"),
        "structured": ("-", "A2")
    },
    "M11": {
        "generic": ("D1 D2", "A1 A2 A3"),
        "structured": ("D1 D2", "A1 A2 A3")
    },
    "M12": {
        "generic": ("-", "A1 A2 A3"),
        "structured": ("D1 -", "A1 A2 A3")
    },
    "M13": {
        "generic": ("- D1:partial", "A3 A4 A5 A6"),
        "structured": ("", "A1 A2 A3 A4 A5 A6")
    },
    "M14": {
        "generic": ("- -", "A1"),
        "structured": ("D1", "A1 A2")
    },
    "M15": {
        "generic": ("- D1", "A1 A2"),
        "structured": ("- - - D1", "A1 A2")
    },
    "M16": {
        "generic": ("- D1", "A1"),
        "structured": ("D1", "A1")
    }
}

review_notes = {
    "M05": "Quality-review decision omits the explicit no-rushing requirement.",
    "M06": "Several extracted decisions remain proposals under the reference rules.",
    "M07": "Promotion decision omits safeguards; CRM is not an accepted reference decision.",
    "M09": "Preparing a reporting format only partially expresses the reporting-policy decision.",
    "M10": "Tracking-system commitment is classified as an action in the reference.",
    "M12": "Using feedback is not equivalent to adopting practical exercises and materials.",
    "M13": "Generic training decision omits the before-implementation requirement.",
    "M14": "Workshops and teaching-material suggestions are not confirmed reference decisions.",
    "M15": "Safety suggestions are unmatched against the current reference decisions.",
    "M16": "Expense control is unmatched against the current reference decision."
}

prepared = []
review_rows = []

# Validate the whole batch before writing.
for mid, prompts in batch_maps.items():
    cid = meeting_register.loc[
        meeting_register["meeting_id"] == mid, "conversation_id"
    ].iloc[0]

    reference = json.loads(
        (DIRS["annotations"] / f"{cid}_reference_reviewed.json")
        .read_text(encoding="utf-8")
    )

    for prompt, (decision_map, action_map) in prompts.items():
        output = json.loads(
            (DIRS[f"{prompt}_outputs"] / f"{mid}_{prompt}_run1.json")
            .read_text(encoding="utf-8")
        )
        units = json.loads(
            (DIRS["evaluation_evidence"]
             / f"{mid}_{prompt}_task_units.json")
            .read_text(encoding="utf-8")
        )

        decisions = [
            {**item, "evaluation_id": f"D{i}"}
            for i, item in enumerate(output["decisions"], 1)
        ]
        actions = [
            item for item in units["action_items"]
            if not item["exclude_duplicate"]
        ]

        data = {
            "meeting_id": mid,
            "conversation_id": cid,
            "prompt": prompt,
            "status": "AI_assisted_judgments_pending_human_review",
            "review_note": review_notes.get(mid, ""),
            "decisions": [],
            "action_items": []
        }

        for key, items, mapping in [
            ("decisions", decisions, decision_map),
            ("action_items", actions, action_map)
        ]:
            tokens = mapping.split()
            if len(tokens) != len(items):
                raise ValueError(
                    f"{mid} {prompt} {key}: unexpected item count."
                )

            valid_ids = {item["id"] for item in reference[key]}
            assigned = []

            for item, token in zip(items, tokens):
                if token == "-":
                    ref_id, judgment = "", "unmatched"
                    reason = (
                        "No equivalent item in the current reference; "
                        "this alone does not establish hallucination."
                    )
                else:
                    parts = token.split(":")
                    ref_id = parts[0]
                    judgment = parts[1] if len(parts) > 1 else "full"
                    if ref_id not in valid_ids:
                        raise ValueError(f"Invalid reference ID: {ref_id}")
                    assigned.append(ref_id)
                    reason = (
                        "Equivalent task content; fields evaluated separately."
                        if judgment == "full"
                        else review_notes.get(
                            mid, "Only part of the reference task is covered."
                        )
                    )

                entry = {
                    "output_id": item["evaluation_id"],
                    "output_task": item["task"],
                    "reference_id": ref_id,
                    "match_judgment": judgment,
                    "reason": reason
                }
                data[key].append(entry)

                target = next(
                    (r["task"] for r in reference[key]
                     if r["id"] == ref_id), ""
                )
                review_rows.append({
                    "input": mid,
                    "prompt": prompt,
                    "type": key,
                    **entry,
                    "reference_task": target
                })

            if len(assigned) != len(set(assigned)):
                raise ValueError("Reference assigned more than once.")

        prepared.append((mid, prompt, data))

for mid, prompt, data in prepared:
    path = (
        DIRS["evaluation_evidence"]
        / f"{mid}_{prompt}_batch_proposed_matches.json"
    )
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

batch_review_table = pd.DataFrame(review_rows)
batch_review_table.to_csv(
    DIRS["evaluation_evidence"] / "batch_match_review.csv",
    index=False
)

print(f"✅ {len(prepared)} proposed match sets saved.")
print("Original match files preserved.")
print("These proposals still require human review.")

In [ ]:
pd.set_option("display.max_colwidth", None)

display(widgets.HTML(
    "<p>Check each output against its reference task. "
    "Full matches must preserve the complete task meaning. "
    "Owner and timing errors are checked separately.</p>"
))

batch_review_display = widgets.Output(
    layout=widgets.Layout(height="500px", overflow="auto")
)

with batch_review_display:
    display(batch_review_table[
        ["input", "prompt", "type", "output_id",
         "output_task", "reference_id", "reference_task",
         "match_judgment", "reason"]
    ])

display(batch_review_display)

In [ ]:
reviewer = input("Reviewer name: ").strip()
confirmed = input(
    "Have you checked all 28 proposed match sets and agree "
    "with their judgments? yes/no: "
).strip().lower()

if reviewer and confirmed == "yes":
    for mid, prompts in batch_maps.items():
        for prompt in prompts:
            path = (
                DIRS["evaluation_evidence"]
                / f"{mid}_{prompt}_batch_proposed_matches.json"
            )
            data = json.loads(path.read_text(encoding="utf-8"))

            data["status"] = "human_reviewed"
            data["reviewer"] = reviewer
            data["reviewed_at"] = datetime.now(timezone.utc).isoformat()

            path.write_text(
                json.dumps(data, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )

    print("✅ All 28 batch match reviews recorded.")
else:
    print("Review remains pending. Send any corrections.")

In [ ]:
task_rows = []
calculation_evidence = []

for _, row in meeting_register.iterrows():
    mid = row["meeting_id"]
    cid = row["conversation_id"]

    reference = json.loads(
        (DIRS["annotations"] / f"{cid}_reference_reviewed.json")
        .read_text(encoding="utf-8")
    )

    for prompt in ["generic", "structured"]:
        if mid == "M01":
            match_name = f"{mid}_{prompt}_corrected_matches.json"
            run = "corrected_run1"
        elif mid == "M02":
            match_name = f"{mid}_{prompt}_matches.json"
            run = "run1"
        else:
            match_name = f"{mid}_{prompt}_batch_proposed_matches.json"
            run = "run1"

        matches = json.loads(
            (DIRS["evaluation_evidence"] / match_name)
            .read_text(encoding="utf-8")
        )

        if matches["status"] != "human_reviewed":
            raise ValueError(f"{mid} {prompt}: review missing.")

        for key, item_type in [
            ("decisions", "decision"),
            ("action_items", "action")
        ]:
            items = matches[key]
            full = [
                item for item in items
                if item["match_judgment"] == "full"
            ]
            full_ids = [item["reference_id"] for item in full]
            reference_ids = {item["id"] for item in reference[key]}

            if (
                len(full_ids) != len(set(full_ids))
                or not set(full_ids).issubset(reference_ids)
            ):
                raise ValueError("Invalid or duplicate reference matches.")

            tp = len(full)
            fp = len(items) - tp
            fn = len(reference[key]) - tp
            partial = sum(
                item["match_judgment"] == "partial" for item in items
            )

            task_rows.append({
                "meeting_id": mid,
                "conversation_id": cid,
                "input_type": row["input_type"],
                "run": run,
                "prompt": prompt,
                "item_type": item_type,
                "predicted_count": len(items),
                "reference_count": len(reference[key]),
                "TP": tp,
                "FP_not_fully_matched": fp,
                "FN": fn,
                "partial_count": partial,
                "precision_percent": (
                    round(100 * tp / (tp + fp), 2)
                    if tp + fp else None
                ),
                "recall_percent": (
                    round(100 * tp / (tp + fn), 2)
                    if tp + fn else None
                ),
                "f1_percent": (
                    round(200 * tp / (2 * tp + fp + fn), 2)
                    if 2 * tp + fp + fn else None
                ),
                "status": "human_reviewed"
            })

            calculation_evidence.append({
                "meeting_id": mid,
                "prompt": prompt,
                "item_type": item_type,
                "run": run,
                "match_file": match_name,
                "full_match_output_ids": [
                    item["output_id"] for item in full
                ],
                "missed_reference_ids": sorted(
                    reference_ids - set(full_ids)
                ),
                "precision_calculation": f"{tp} / {tp + fp} * 100",
                "recall_calculation": f"{tp} / {tp + fn} * 100",
                "f1_calculation": (
                    f"{2 * tp} / {2 * tp + fp + fn} * 100"
                )
            })

all_task_metrics = pd.DataFrame(task_rows)

all_task_metrics.to_csv(
    DIRS["results"] / "all_input_task_metrics.csv",
    index=False
)

(DIRS["evaluation_evidence"] / "task_calculations.json").write_text(
    json.dumps(calculation_evidence, indent=2),
    encoding="utf-8"
)

display(all_task_metrics)
print("✅ 64 task-metric rows saved with calculation evidence.")

In [ ]:
import re

def field_text(value):
    value = str(value).lower().strip()
    value = re.sub(r"[^\w\s]", " ", value)
    return " ".join(value.split())

field_rows = []

for _, row in meeting_register.iterrows():
    mid = row["meeting_id"]
    if mid == "M01":
        continue  # Corrected M01 fields already evaluated.

    reference = json.loads(
        (DIRS["annotations"]
         / f"{row['conversation_id']}_reference_reviewed.json")
        .read_text(encoding="utf-8")
    )
    targets = {
        item["id"]: item for item in reference["action_items"]
    }

    for prompt in ["generic", "structured"]:
        match_name = (
            f"{mid}_{prompt}_matches.json"
            if mid == "M02"
            else f"{mid}_{prompt}_batch_proposed_matches.json"
        )
        matches = json.loads(
            (DIRS["evaluation_evidence"] / match_name)
            .read_text(encoding="utf-8")
        )
        units = json.loads(
            (DIRS["evaluation_evidence"]
             / f"{mid}_{prompt}_task_units.json")
            .read_text(encoding="utf-8")
        )
        predictions = {
            item["evaluation_id"]: item
            for item in units["action_items"]
            if not item["exclude_duplicate"]
        }

        for match in matches["action_items"]:
            if match["match_judgment"] != "full":
                continue

            predicted = predictions[match["output_id"]]
            target = targets[match["reference_id"]]

            for field in ["owner", "deadline", "condition"]:
                ambiguous = (
                    field == "owner"
                    and target.get("owner_ambiguous", False)
                )
                same = (
                    field_text(predicted[field])
                    == field_text(target[field])
                )

                field_rows.append({
                    "meeting_id": mid,
                    "conversation_id": row["conversation_id"],
                    "prompt": prompt,
                    "output_id": match["output_id"],
                    "reference_id": match["reference_id"],
                    "field": field,
                    "reference_value": target[field],
                    "predicted_value": predicted[field],
                    "judgment": (
                        "excluded_ambiguous" if ambiguous
                        else "proposed_correct" if same
                        else "needs_semantic_review"
                    ),
                    "reason": (
                        "Reference owner is ambiguous."
                        if ambiguous else ""
                    ),
                    "status": "pending_review"
                })

batch_field_checks = pd.DataFrame(field_rows)

batch_field_checks.to_csv(
    DIRS["evaluation_evidence"] / "batch_field_checks_draft.csv",
    index=False
)

# Plain text avoids copying webpage styling.
flagged = batch_field_checks[
    batch_field_checks["judgment"] == "needs_semantic_review"
]

print(flagged[
    ["meeting_id", "prompt", "output_id", "reference_id",
     "field", "reference_value", "predicted_value"]
].to_string(index=True))

print("\n✅ Batch field checks prepared.")
print("Ambiguous reference owners excluded from owner scoring.")

In [ ]:
equivalent_rows = {
    ("M02", "structured", "A12", "condition"),
    ("M05", "structured", "A14", "condition"),
    ("M06", "generic", "A2", "deadline"),
    ("M16", "generic", "A1", "owner"),
    ("M16", "structured", "A1", "owner"),
}

unassessable_rows = {
    ("M07", "structured", output_id, "deadline")
    for output_id in ["A1", "A2", "A3", "A4"]
}

for index, row in batch_field_checks.iterrows():
    if row["judgment"] == "excluded_ambiguous":
        continue

    key = (
        row["meeting_id"], row["prompt"],
        row["output_id"], row["field"]
    )

    if key in equivalent_rows:
        judgment = "correct"
        reason = "Equivalent wording; honorifics do not change identity."

    elif key in unassessable_rows:
        judgment = "excluded_unspecified_timing"
        reason = (
            "Transcript mentions an agreed timeline but does not "
            "state its value; excluded from deadline scoring."
        )

    elif row["judgment"] == "proposed_correct":
        judgment = "correct"
        reason = "Matches the reference after text normalization."

    else:
        judgment = "incorrect"
        if row["field"] == "condition":
            reason = (
                "Missing conditional timing or temporal wording "
                "incorrectly placed in the condition field."
            )
        elif not row["predicted_value"]:
            reason = "Explicit reference deadline omitted."
        else:
            reason = (
                "Timing is not explicitly assigned to this task "
                "under the annotation rules."
            )

    batch_field_checks.loc[index, "judgment"] = judgment
    batch_field_checks.loc[index, "reason"] = reason

batch_field_checks.to_csv(
    DIRS["evaluation_evidence"] / "batch_field_checks_proposed.csv",
    index=False
)

print(
    batch_field_checks.groupby(
        ["field", "judgment"]
    ).size().to_string()
)
print("\n✅ Proposed semantic judgments saved.")

In [ ]:
reviewer = input("Reviewer name: ").strip()
confirmed = input(
    "Have you reviewed and accepted the batch field judgments? yes/no: "
).strip().lower()

if reviewer and confirmed == "yes":
    batch_field_checks["status"] = "human_reviewed"
    batch_field_checks["reviewer"] = reviewer
    batch_field_checks["reviewed_at"] = (
        datetime.now(timezone.utc).isoformat()
    )

    batch_field_checks.to_csv(
        DIRS["evaluation_evidence"] / "batch_field_checks_reviewed.csv",
        index=False
    )

    # Include the already-reviewed corrected M01 fields.
    m01 = pd.read_csv(
        DIRS["evaluation_evidence"] / "M01_corrected_field_checks.csv",
        keep_default_na=False
    )
    m01["meeting_id"] = "M01"
    m01["conversation_id"] = "C01"
    m01["judgment"] = m01["correct"].map(
        lambda value: (
            "correct" if str(value).lower() == "true" else "incorrect"
        )
    )

    all_field_checks = pd.concat(
        [m01, batch_field_checks], ignore_index=True
    )
    all_field_checks.to_csv(
        DIRS["evaluation_evidence"] / "all_field_checks.csv",
        index=False
    )

    summary_rows = []

    for (mid, prompt, field), group in all_field_checks.groupby(
        ["meeting_id", "prompt", "field"]
    ):
        correct = (group["judgment"] == "correct").sum()
        incorrect = (group["judgment"] == "incorrect").sum()
        excluded = group["judgment"].str.startswith("excluded").sum()
        assessed = correct + incorrect

        summary_rows.append({
            "meeting_id": mid,
            "prompt": prompt,
            "field": field,
            "correct_count": int(correct),
            "assessed_count": int(assessed),
            "excluded_count": int(excluded),
            "accuracy_percent": (
                round(100 * correct / assessed, 2)
                if assessed else None
            ),
            "calculation": f"{correct} / {assessed} * 100"
        })

    all_field_metrics = pd.DataFrame(summary_rows)
    all_field_metrics.to_csv(
        DIRS["results"] / "all_input_field_metrics.csv",
        index=False
    )

    display(all_field_metrics)
    print("✅ Field scores and calculation evidence saved.")
else:
    print("Fields remain provisional. Tell me the corrections.")


In [ ]:
import shutil
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

PROJECT_ROOT = Path("/content/AIMIS_Text_Audio_Evaluation")

if not (PROJECT_ROOT / "meeting_register.csv").exists():
    raise FileNotFoundError("Project files are missing. Restore your backup first.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

zip_path = shutil.make_archive(
    f"/content/AIMIS_FINAL_Backup_{stamp}",
    "zip",
    root_dir=str(PROJECT_ROOT.parent),
    base_dir=PROJECT_ROOT.name
)

files.download(zip_path)
print("✅ ZIP ready. Keep the downloaded file.")